In [1]:
from datasets import load_dataset
dataset = load_dataset("zillow/real_estate_v1")

In [2]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['messages', 'split', 'topic'],
        num_rows: 21382
    })
})


In [3]:
def flatten_conversation(example):
    full_text = " ".join([m["content"] for m in example["messages"]])
    return {"text": full_text}

dataset = dataset["train"].map(flatten_conversation)

In [4]:
dataset[0]["text"]

"Hi, I'm considering moving to a new neighborhood and I'm really curious about the nightlife and entertainment options available. Can you help me with that? Absolutely, I'd be happy to help you with information about nightlife and entertainment options in your potential new neighborhood. To start, could you tell me a bit more about what kind of nightlife and entertainment you enjoy? For example, are you interested in bars, clubs, live music venues, theaters, or something else? I'm mostly interested in live music venues and good bars where I can hang out with friends. I also enjoy going to the theater occasionally. Great choices! Let’s break it down a bit. \n\nFor live music venues, many neighborhoods have a variety of options ranging from small, intimate settings to larger concert halls. These venues often host local bands as well as touring artists. If you tell me the neighborhood you’re considering, I can provide details on specific venues.\n\nAs for bars, neighborhoods typically hav

In [5]:
from keybert import KeyBERT

kw_model = KeyBERT("all-MiniLM-L6-v2")

def generate_keywords(example):
    keywords = kw_model.extract_keywords(example["text"], top_n=20)
    example["keywords"] = [kw[0] for kw in keywords]
    return example

dataset = dataset.select(range(1000))  # limit for speed
dataset = dataset.map(generate_keywords)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [13]:
dataset[0]["keywords"]

['venues', 'nightlife', 'neighborhoods', 'neighborhood', 'concerts']

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [16]:
label_map = {"O": 0, "B-KEY": 1, "I-KEY": 2}

def tokenize_and_align_labels(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
        return_offsets_mapping=True
    )
    
    labels = [0] * len(tokens["input_ids"])
    
    text = example["text"].lower()
    
    for keyword in example["keywords"]:
        keyword = keyword.lower()
        start = text.find(keyword)
        if start == -1:
            continue
        end = start + len(keyword)
        
        for idx, (offset_start, offset_end) in enumerate(tokens["offset_mapping"]):
            if offset_start >= start and offset_end <= end:
                if offset_start == start:
                    labels[idx] = label_map["B-KEY"]
                else:
                    labels[idx] = label_map["I-KEY"]
    
    tokens["labels"] = labels
    tokens.pop("offset_mapping")
    return tokens

In [17]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [19]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [45]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [46]:
trainer.train()

Step,Training Loss
50,0.048400
100,0.039800
150,0.036800
200,0.030000
250,0.031300
300,0.022400
350,0.021600


TrainOutput(global_step=375, training_loss=0.03226542965571086, metrics={'train_runtime': 61.9942, 'train_samples_per_second': 48.392, 'train_steps_per_second': 6.049, 'total_flos': 195983193600000.0, 'train_loss': 0.03226542965571086, 'epoch': 3.0})

In [22]:
predictions = trainer.predict(tokenized_dataset)

In [48]:
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

label_list = ["O", "B-KEY", "I-KEY"]

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [49]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.2)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [50]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

In [51]:
trainer.train()

Step,Training Loss
50,0.026100
100,0.024100
150,0.016700
200,0.015300
250,0.011200
300,0.010900


TrainOutput(global_step=300, training_loss=0.017362803419431052, metrics={'train_runtime': 49.5084, 'train_samples_per_second': 48.477, 'train_steps_per_second': 6.06, 'total_flos': 156786554880000.0, 'train_loss': 0.017362803419431052, 'epoch': 3.0})

In [52]:
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.01873059943318367, 'eval_precision': 0.8345991561181435, 'eval_recall': 0.8822479928635147, 'eval_f1': 0.8577623590633131, 'eval_accuracy': 0.99330078125, 'eval_runtime': 1.9925, 'eval_samples_per_second': 100.375, 'eval_steps_per_second': 12.547, 'epoch': 3.0}


In [53]:
model.eval()

DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

In [54]:
import torch

text = """
I'm looking for a quiet neighborhood near downtown Seattle 
with good schools and affordable housing.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

In [55]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

inputs = {k: v.to(device) for k, v in inputs.items()}

In [56]:
with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

In [57]:
label_list = ["O", "B-KEY", "I-KEY"]

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
pred_labels = [label_list[p.item()] for p in predictions[0]]

In [59]:
keywords = []
current_keyword = []

for token, label in zip(tokens, pred_labels):
    print(token, label)

for token, label in zip(tokens, pred_labels):
    if token in ["[CLS]", "[SEP]", "[PAD]"]:
        continue

    if label == "B-KEY":
        if current_keyword:
            keywords.append(" ".join(current_keyword))
            current_keyword = []
        current_keyword.append(token)

    elif label == "I-KEY":
        current_keyword.append(token)

    else:
        if current_keyword:
            keywords.append(" ".join(current_keyword))
            current_keyword = []

if current_keyword:
    keywords.append(" ".join(current_keyword))

# Clean subword tokens (##)
cleaned_keywords = []
for kw in keywords:
    cleaned = kw.replace(" ##", "")
    cleaned_keywords.append(cleaned)

print("Extracted keywords:")
for kw in cleaned_keywords:
    print("-", kw)

[CLS] O
i O
' O
m O
looking O
for O
a O
quiet O
neighborhood O
near O
downtown O
seattle O
with O
good O
schools O
and O
affordable O
housing O
. O
[SEP] O
Extracted keywords:
